# Session 2: Function Reform (Policy Function)

Goal: replace one policy function in the tax-transfer system and verify that the reform actually worked.

Purpose of this exercise: override the Günstigerprüfung by forcing `kinderfreibetrag_günstiger_sn` to always return `False`, rerun the model, and compare the tax effect to the baseline.

This notebook uses `@policy_function()` (output logic), not `@param_function()` (parameter construction).

Reference how-to guide (full version):
https://gettsim.readthedocs.io/en/stable/how_to_guides/modifications_of_policy_environments.html

## 0) Setup

In [ ]:
# Command for installing gettsim (only use on Colab)

# !apt-get update -qq
# !apt-get install -y graphviz graphviz-dev pkg-config
# !pip install pygraphviz
# !pip install gettsim
# !pip install git+https://github.com/ttsim-dev/gettsim-personas.git

In [16]:
import numpy as np
import pandas as pd

from gettsim import InputData, MainTarget, TTTargets, copy_environment, main
from gettsim.tt import policy_function

In [17]:
POLICY_DATE = "2025-01-01"

## 1) Baseline run (status quo)

In [18]:
from gettsim_personas import einkommensteuer_sozialabgaben

# We use a coherent persona to avoid input-debugging noise.
persona = einkommensteuer_sozialabgaben.Couple1Child(policy_date_str=POLICY_DATE)

# Raise wages so tax effects are clearly visible in the reform comparison.
persona_high_income = persona.upsert_input_data(
    {"einnahmen": {"bruttolohn_m": np.array([9000.0, 7000.0, 0.0])}}
)

print(persona.description)

Persona to compute income taxes and social insurance contributions. Jointly
        taxed married couple with one child. All transfers are set to zero; don't use
        this persona for low- to mid-income households, as they may be eligible for
        (means-tested) transfers.


In [19]:
# Keep target scope tight so reform effects are easy to interpret.
tt_targets = TTTargets(tree={"einkommensteuer": {"betrag_y_sn": None}, 'kindergeld': {'betrag_m_hh': None}})

In [20]:
# Build status-quo policy environment object.
status_quo_environment = main(
    main_target=MainTarget.policy_environment,
    policy_date_str=POLICY_DATE,
    include_warn_nodes=False,
)

In [21]:
baseline = main(
    main_target=MainTarget.results.df_with_nested_columns,
    policy_date_str=POLICY_DATE,
    input_data=InputData.tree(tree=persona_high_income.input_data_tree),
    tt_targets=tt_targets,
    include_warn_nodes=False,
)

baseline

,einkommensteuer,kindergeld
,betrag_y_sn,betrag_m_hh
p_id,,
0,43792.0,255
1,43792.0,255
2,0.0,255


## 2) Define and inject function reform

Reform rule: override `kinderfreibetrag_günstiger_sn` so that it always returns `False`.

In [22]:
# Always copy before modifying, so status quo remains unchanged.
reform_environment = copy_environment(status_quo_environment)

In [23]:
@policy_function()
def kinderfreibetrag_günstiger_sn(
    betrag_ohne_kinderfreibetrag_y_sn: float,
    betrag_mit_kinderfreibetrag_y_sn: float,
    relevantes_kindergeld_y_sn: float,
) -> bool:
    # Reform: always choose Kindergeld branch in final tax selection.
    return False

In [24]:
# Replace status-quo function with our reform function.
reform_environment["einkommensteuer"]["kinderfreibetrag_günstiger_sn"] = kinderfreibetrag_günstiger_sn

## 3) Structural verification (did replacement happen?)

In [25]:
print("Status-quo function object:")
print(status_quo_environment["einkommensteuer"]["kinderfreibetrag_günstiger_sn"])

print("\nReform function object:")
print(reform_environment["einkommensteuer"]["kinderfreibetrag_günstiger_sn"])

Status-quo function object:
PolicyFunction(leaf_name='kinderfreibetrag_günstiger_sn', start_date=datetime.date(1900, 1, 1), end_date=datetime.date(2099, 12, 31), description='Kinderfreibetrag more favorable than Kindergeld.', function=<function kinderfreibetrag_günstiger_sn at 0x14780df30>, rounding_spec=None, foreign_key_type=<FKType.IRRELEVANT: 'irrelevant'>, warn_msg_if_included=None, fail_msg_if_included=None, vectorization_strategy='vectorize')

Reform function object:
PolicyFunction(leaf_name='kinderfreibetrag_günstiger_sn', start_date=datetime.date(1900, 1, 1), end_date=datetime.date(2099, 12, 31), description='None', function=<function kinderfreibetrag_günstiger_sn at 0x14673f480>, rounding_spec=None, foreign_key_type=<FKType.IRRELEVANT: 'irrelevant'>, warn_msg_if_included=None, fail_msg_if_included=None, vectorization_strategy='vectorize')


## 4) Re-run with reform environment

In [26]:
reform = main(
    main_target=MainTarget.results.df_with_nested_columns,
    policy_date_str=POLICY_DATE,
    input_data=InputData.tree(tree=persona_high_income.input_data_tree),
    tt_targets=tt_targets,
    policy_environment=reform_environment,
    include_warn_nodes=False,
)

reform

,einkommensteuer,kindergeld
,betrag_y_sn,betrag_m_hh
p_id,,
0,44764.0,255
1,44764.0,255
2,0.0,255


In [29]:
reform - baseline

,einkommensteuer,kindergeld
,betrag_y_sn,betrag_m_hh
p_id,,
0,972.0,0
1,972.0,0
2,0.0,0
